In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import missingno as msno
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.api as sm
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.svm import SVR
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from scipy.stats import ttest_rel
from sklearn.compose import TransformedTargetRegressor

In [2]:
df_lof = pd.read_csv('lof.csv')

In [3]:
X = df_lof.drop(['Dev Time (Days)', 'issue_key','Current Status'], axis=1)
y = np.log1p(df_lof['Dev Time (Days)'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

print("Model trained successfully!")
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
print(f"Mean Absolute Error (MAE): {mae:.2f}")

r2 = r2_score(y_test, predictions)
print(f"R-squared Score: {r2:.2f}")

imp = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(imp.to_string(max_rows=None))

Model trained successfully!
Mean Absolute Error (MAE): 0.31
R-squared Score: 0.11
Start Date                                  0.525317
total_swag                                  0.036863
fix_version_(no version)                    0.022723
impacted_area_Blackbox Cloud                0.020348
impacted_area_SPG                           0.020158
impacted_area_(none)                        0.019575
impacted_area_Low Level                     0.015198
impacted_area_SWBB Android Core / Common    0.014192
impacted_area_Box Test                      0.014029
impacted_area_Orderability                  0.013104
impacted_area_CPE-EE                        0.012124
impacted_area_Encryption                    0.011748
technology_APX                              0.011028
impacted_area_E2E                           0.010247
impacted_area_IT                            0.009711
impacted_area_RC Insights                   0.009591
impacted_area_Radio Central                 0.009494
technology_(none)

# Test Multiple Imputation Methods for total_swag (After Isolation Forest)

In [5]:
df = df_lof.copy()
num_cols = df.select_dtypes(include='number').columns
assert 'total_swag' in num_cols, "total_swag must be numeric."
df_num = df[num_cols]

In [6]:
obs_idx = df_num['total_swag'].dropna().index
rng = np.random.default_rng(42)
val_size = max(1, int(0.2 * len(obs_idx)))
val_idx = rng.choice(obs_idx, size=val_size, replace=False)
df_test = df_num.copy()
y_true = df_test.loc[val_idx, 'total_swag'].to_numpy()
df_test.loc[val_idx, 'total_swag'] = np.nan

Bayesian Ridge (features scaled)

In [7]:
est = make_pipeline(StandardScaler(), BayesianRidge())
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=est, sample_posterior = True, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 679.237 RMSE: 1512425.135 Mean imputation SD: 1026.670


Linear Regression

In [8]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=LinearRegression(), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 5430.566 RMSE: 48615120.342 Mean imputation SD: 0.000


Decision Tree

In [9]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=DecisionTreeRegressor(random_state=seed), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 569.756 RMSE: 1023731.856 Mean imputation SD: 167.778


Random Forest

In [10]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=RandomForestRegressor(n_estimators=200, random_state=seed, n_jobs=-1), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 524.961 RMSE: 958243.768 Mean imputation SD: 25.549


Extra Trees

In [11]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=ExtraTreesRegressor(n_estimators=200, random_state=seed, n_jobs=-1), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 521.688 RMSE: 1114439.078 Mean imputation SD: 19.327


Gradient boosting

In [12]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=GradientBoostingRegressor(random_state=seed), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 512.907 RMSE: 900438.291 Mean imputation SD: 24.460


K Neighbours

In [13]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=KNeighborsRegressor(n_neighbors=5), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 787.474 RMSE: 1798752.366 Mean imputation SD: 0.000


Choose Gradient Boosting

In [15]:
use_lof = df_lof.drop(['issue_key', 'Current Status'], axis=1)

imputer_lof = IterativeImputer(
    estimator = GradientBoostingRegressor(random_state=42),
    max_iter = 10,
    random_state = 42
)

df_lof_imputed_values = imputer_lof.fit_transform(use_lof)
df_lof_final = pd.DataFrame(df_lof_imputed_values, columns=use_lof.columns, index=use_lof.index)

In [16]:
df_lof_final.to_csv('imputed_lof.csv', index=False)